# Prompt 30: Pandas API on Apache Spark (pyspark.pandas)
## Databricks Certified Associate Developer for Apache Spark
### Topic 7 — Using Pandas API on Apache Spark (5%)

---

## What is the Pandas API on Apache Spark?

The **Pandas API on Apache Spark** (module `pyspark.pandas`) is a **pandas-compatible API** that runs on Spark, enabling pandas-style code to process **distributed data** at scale. It was previously an independent project called **Koalas** (by Databricks), which was merged into PySpark as of **Spark 3.2**.

**Core idea:** Write code that looks like pandas, but runs distributed across a Spark cluster.

```
NATIVE PANDAS               PANDAS API ON SPARK          NATIVE PYSPARK
─────────────────           ─────────────────────        ─────────────────────
import pandas as pd         import pyspark.pandas as ps  from pyspark.sql import *

df = pd.read_csv(...)       df = ps.read_csv(...)        df = spark.read.csv(...)

df.groupby('col')           df.groupby('col')            df.groupBy('col')
  .agg({'val': 'sum'})        .agg({'val': 'sum'})          .agg(sum('val'))

Single-machine only         Distributed (Spark)          Distributed (Spark)
In-memory (RAM limit)       Unlimited scale              Unlimited scale
Familiar pandas syntax      Familiar pandas syntax       Spark-specific syntax
```

---

## Import Pattern (Exam Critical)

```python
# CORRECT — Pandas API on Spark
import pyspark.pandas as ps

# WRONG — this is native pandas (single-machine)
import pandas as pd

# Also acceptable (same thing, older alias)
import pyspark.pandas as ps
# ps is the conventional alias — the exam tests this import
```

---

## Creating a Spark Pandas DataFrame

```python
import pyspark.pandas as ps

# 1. From Python dict/list (same as pandas)
psdf = ps.DataFrame({
    'name': ['Alice', 'Bob', 'Carol'],
    'age': [30, 25, 35],
    'salary': [95000, 72000, 88000]
})

# 2. From a CSV file (reads in distributed manner on Spark)
psdf = ps.read_csv('/data/employees.csv')

# 3. From a native pandas DataFrame
import pandas as pd
pandas_df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
psdf = ps.from_pandas(pandas_df)  # converts pandas → Spark pandas

# 4. From a Spark DataFrame
spark_df = spark.read.parquet('/data/sales.parquet')
psdf = spark_df.pandas_api()       # Spark DataFrame → Spark pandas
# OR:
psdf = ps.from_spark(spark_df)     # same result
```

---

## Type Conversion Methods (Exam Critical)

```
                    ps.from_pandas(pdf)
Native pandas  ─────────────────────────────►  Spark pandas (pyspark.pandas)
    (pdf)            psdf.to_pandas()          ◄─────────────────────────────

                    spark_df.pandas_api()
Spark DataFrame ────────────────────────────►  Spark pandas (pyspark.pandas)
  (spark_df)       OR ps.from_spark(spark_df) ◄────────────────────────────
                          psdf.to_spark()                   psdf.to_spark()
```

| Conversion | Method | Warning |
|-----------|--------|------|
| Spark pandas → Native pandas | `psdf.to_pandas()` | **Full collect** — all data to driver memory. Only for small DataFrames! |
| Native pandas → Spark pandas | `ps.from_pandas(pdf)` | Data is distributed to Spark cluster |
| Spark DataFrame → Spark pandas | `sdf.pandas_api()` | No data movement — changes API only |
| Spark pandas → Spark DataFrame | `psdf.to_spark()` | No data movement — changes API only |

---

## Key Behavioral Differences from Native pandas

### 1. Index behavior
```python
# Native pandas: integer RangeIndex, stable and always sequential
pdf.index  # RangeIndex(start=0, stop=5, step=1)

# Spark pandas: default index may be distributed/unstable
# Options via spark.conf:
#   'sequence'       — sequential integers (requires full scan, slow)
#   'distributed'    — distributed integers (fast but NOT globally sequential)
#   'distributed-sequence' — globally sequential but requires extra shuffle
ps.set_option('compute.default_index_type', 'distributed')  # fastest option
```

### 2. No guaranteed row ordering
```python
# Native pandas: rows are ordered by insertion order
pdf.head(3)  # always returns first 3 rows in insertion order

# Spark pandas: distributed — no guaranteed order without explicit sort
psdf.head(3)  # returns 3 rows but order may vary
psdf.sort_values('salary').head(3)  # explicit sort required for determinism
```

### 3. No in-place operations
```python
# Native pandas: in-place supported
pdf.drop('col', axis=1, inplace=True)  # modifies pdf in place

# Spark pandas: in-place NOT supported (DataFrames are immutable in Spark)
psdf = psdf.drop('col', axis=1)  # must reassign
```

### 4. Some operations are expensive or unsupported
```python
# These trigger a full collect/shuffle and are expensive:
psdf.sort_values('col')        # requires global sort → full shuffle
len(psdf)                      # triggers count action
psdf.iterrows()                # iterates row-by-row — extremely slow
psdf.iteritems()               # same issue

# Preferred alternatives:
psdf.sort_values('col').head(n)  # still needs sort but bounded output
psdf.shape[0]                    # same as len() — still triggers count
# Avoid row-level iteration entirely — use vectorised operations
```

### 5. Column access syntax differences
```python
# Both native pandas and Spark pandas support:
psdf['col']       # column access by key
psdf[['a', 'b']] # multi-column select
psdf.col          # attribute access (if col is not a method name)
```

In [ ]:
# Cell 1: Setup — SparkSession needed for pyspark.pandas
# pyspark.pandas uses the active SparkSession automatically

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName('PandasAPIOnSpark') \
    .master('local[*]') \
    .config('spark.sql.shuffle.partitions', '4') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

# Import pyspark.pandas — uses the active SparkSession automatically
import pyspark.pandas as ps
import pandas as pd
import numpy as np

print('=== Setup complete ===')
print(f'Spark version: {spark.version}')
print(f'pyspark.pandas version: {ps.__version__}')
print()
print('Convention: ps  = pyspark.pandas  (Pandas API on Spark)')
print('Convention: pd  = pandas           (native pandas)')
print('Convention: psdf = Spark pandas DataFrame')
print('Convention: pdf  = native pandas DataFrame')

In [ ]:
# Cell 2: Creating Spark pandas DataFrames — all four methods

print('=== Method 1: ps.DataFrame() from Python dict ===')
psdf_from_dict = ps.DataFrame({
    'name':     ['Alice', 'Bob', 'Carol', 'Dave', 'Eve'],
    'dept':     ['Engineering', 'Marketing', 'Engineering', 'HR', 'Marketing'],
    'salary':   [95000, 72000, 88000, 62000, 78000],
    'years':    [5, 3, 7, 2, 4],
})
print(psdf_from_dict)
print(f'Type: {type(psdf_from_dict)}')
print()

print('=== Method 2: ps.from_pandas() — convert native pandas → Spark pandas ===')
pdf_native = pd.DataFrame({
    'product':  ['Laptop', 'Phone', 'Tablet', 'Watch'],
    'category': ['Electronics', 'Electronics', 'Electronics', 'Accessories'],
    'price':    [1499.99, 799.99, 599.99, 299.99],
    'stock':    [10, 25, 8, 50],
})
psdf_from_pandas = ps.from_pandas(pdf_native)
print(f'Native pandas type: {type(pdf_native)}')
print(f'Spark pandas type:  {type(psdf_from_pandas)}')
print(psdf_from_pandas)
print()

print('=== Method 3: spark_df.pandas_api() — Spark DataFrame → Spark pandas ===')
spark_df = spark.createDataFrame([
    (1, 'Widget A', 'Parts', 12.50),
    (2, 'Widget B', 'Parts', 18.75),
    (3, 'Gadget X', 'Tools', 34.99),
], ['id', 'item', 'category', 'cost'])

psdf_from_spark = spark_df.pandas_api()
print(f'Spark DataFrame type: {type(spark_df)}')
print(f'Spark pandas type:    {type(psdf_from_spark)}')
print(psdf_from_spark)
print()

print('=== Method 4: ps.from_spark() — alternative to .pandas_api() ===')
psdf_from_spark2 = ps.from_spark(spark_df)
print(f'Type: {type(psdf_from_spark2)}')
print(psdf_from_spark2)

In [ ]:
# Cell 3: Type conversions — Spark pandas ↔ native pandas ↔ Spark DataFrame

print('=== Conversion: Spark pandas → native pandas (to_pandas()) ===')
psdf_small = ps.DataFrame({'x': [1, 2, 3], 'y': ['a', 'b', 'c']})
pdf_converted = psdf_small.to_pandas()
print(f'Spark pandas type: {type(psdf_small)}')
print(f'Native pandas type: {type(pdf_converted)}')
print(pdf_converted)
print()
print('WARNING: .to_pandas() collects ALL data to the driver.')
print('Only call on small DataFrames — will OOM on large distributed data!')
print()

print('=== Conversion: Spark pandas → Spark DataFrame (to_spark()) ===')
psdf_employees = ps.DataFrame({
    'name':   ['Alice', 'Bob', 'Carol'],
    'salary': [95000, 72000, 88000],
    'dept':   ['Eng', 'Mkt', 'Eng'],
})
sdf_back = psdf_employees.to_spark()
print(f'Spark pandas type:  {type(psdf_employees)}')
print(f'Spark DataFrame type: {type(sdf_back)}')
sdf_back.printSchema()
sdf_back.show()

# Conversion summary table
print('=== Conversion Reference ===')
conversions = [
    ('Native pandas  → Spark pandas',  'ps.from_pandas(pdf)'),
    ('Spark pandas   → Native pandas', 'psdf.to_pandas()  [CAUTION: full collect]'),
    ('Spark DataFrame→ Spark pandas',  'sdf.pandas_api()  OR  ps.from_spark(sdf)'),
    ('Spark pandas   → Spark DataFrame','psdf.to_spark()'),
]
for conv, method in conversions:
    print(f'  {conv:<40} {method}')

In [ ]:
# Cell 4: Side-by-side comparison — same operation in 3 APIs
# native pandas | pyspark.pandas | native PySpark DataFrame

from pyspark.sql.functions import col, sum as _sum, avg, count, desc

# Source data — same dataset for all three approaches
raw_data = [
    ('Alice',  'Engineering', 95000, 5),
    ('Bob',    'Marketing',   72000, 3),
    ('Carol',  'Engineering', 88000, 7),
    ('Dave',   'HR',          62000, 2),
    ('Eve',    'Marketing',   78000, 4),
    ('Frank',  'Engineering', 105000, 8),
    ('Grace',  'HR',          58000, 1),
    ('Hank',   'Marketing',   75000, 6),
]
columns = ['name', 'department', 'salary', 'years']

# ----------------------------------------------------------------
# API 1: Native pandas
# ----------------------------------------------------------------
pdf = pd.DataFrame(raw_data, columns=columns)

print('=== NATIVE PANDAS: groupby → agg → sort ===')
pdf_result = pdf.groupby('department').agg(
    headcount=('name', 'count'),
    avg_salary=('salary', 'mean'),
    total_salary=('salary', 'sum')
).reset_index().sort_values('total_salary', ascending=False)
print(pdf_result.to_string(index=False))
print()

# ----------------------------------------------------------------
# API 2: pyspark.pandas (Pandas API on Spark)
# ----------------------------------------------------------------
psdf = ps.from_pandas(pdf)

print('=== PYSPARK.PANDAS (Pandas API on Spark): groupby → agg → sort ===')
psdf_result = psdf.groupby('department').agg(
    headcount=('name', 'count'),
    avg_salary=('salary', 'mean'),
    total_salary=('salary', 'sum')
).reset_index().sort_values('total_salary', ascending=False)
print(psdf_result.to_string())
print(f'Type: {type(psdf_result)}')
print()

# ----------------------------------------------------------------
# API 3: Native PySpark DataFrame API
# ----------------------------------------------------------------
sdf = spark.createDataFrame(raw_data, columns)

print('=== NATIVE PYSPARK: groupBy → agg → orderBy ===')
sdf_result = sdf.groupBy('department') \
    .agg(
        count('name').alias('headcount'),
        avg('salary').alias('avg_salary'),
        _sum('salary').alias('total_salary')
    ) \
    .orderBy(desc('total_salary'))
sdf_result.show()
print(f'Type: {type(sdf_result)}')

In [ ]:
# Cell 5: Common pyspark.pandas operations — the exam-relevant ones

# Sample dataset
psdf = ps.DataFrame({
    'name':       ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank', 'Grace'],
    'department': ['Eng', 'Mkt', 'Eng', 'HR', 'Mkt', 'Eng', 'HR'],
    'salary':     [95000, 72000, 88000, 62000, 78000, 105000, 58000],
    'years':      [5, 3, 7, 2, 4, 8, 1],
    'rating':     [4.5, 3.8, 4.2, 3.5, 4.0, 4.8, 3.2],
})

# -------------------------------------------------------
# 1. describe() — statistical summary
# -------------------------------------------------------
print('=== 1. describe() — statistical summary ===')
print(psdf.describe())

# -------------------------------------------------------
# 2. value_counts() — frequency of each value
# -------------------------------------------------------
print('\n=== 2. value_counts() — department frequency ===')
print(psdf['department'].value_counts())

# -------------------------------------------------------
# 3. groupby + agg with dict
# -------------------------------------------------------
print('\n=== 3. groupby().agg() ===')
result = psdf.groupby('department').agg({'salary': ['mean', 'min', 'max'], 'years': 'mean'})
print(result)

# -------------------------------------------------------
# 4. merge() — equivalent to pandas merge / Spark join
# -------------------------------------------------------
print('\n=== 4. merge() ===')
psdf_dept = ps.DataFrame({
    'department': ['Eng', 'Mkt', 'HR'],
    'budget':     [500000, 200000, 150000],
    'manager':    ['Alex', 'Jordan', 'Morgan'],
})
merged = psdf.merge(psdf_dept, on='department', how='left')
print(merged[['name', 'department', 'salary', 'manager', 'budget']].sort_values('name'))

# -------------------------------------------------------
# 5. apply() — apply a function to each element / row
# -------------------------------------------------------
print('\n=== 5. apply() — salary tier ===')
def salary_tier(s):
    if s >= 95000: return 'Senior'
    elif s >= 75000: return 'Mid'
    else: return 'Junior'

psdf['tier'] = psdf['salary'].apply(salary_tier)
print(psdf[['name', 'salary', 'tier']].sort_values('salary', ascending=False))

# -------------------------------------------------------
# 6. pivot_table()
# -------------------------------------------------------
print('\n=== 6. pivot_table() ===')
pivot = psdf.pivot_table(
    values='salary',
    index='department',
    aggfunc='mean'
)
print(pivot)

# -------------------------------------------------------
# 7. Filtering — same as pandas
# -------------------------------------------------------
print('\n=== 7. Filtering ===')
high_earners = psdf[psdf['salary'] > 85000]
print(high_earners[['name', 'department', 'salary']])

# -------------------------------------------------------
# 8. Adding a computed column
# -------------------------------------------------------
print('\n=== 8. Computed column ===')
psdf['salary_per_year'] = psdf['salary'] / psdf['years']
print(psdf[['name', 'salary', 'years', 'salary_per_year']].sort_values('salary_per_year', ascending=False))

spark.stop()
print('\nDone.')

## Performance Considerations

### When pyspark.pandas adds overhead vs when it's beneficial

| Situation | Recommendation |
|-----------|----------------|
| Migrating existing pandas code to run on large data | **Use pyspark.pandas** — minimal code change required |
| New code for large-scale data processing | **Use native PySpark DataFrame API** — more control, better performance |
| Data fits in a single machine's RAM (< a few GB) | **Use native pandas** — no distributed overhead |
| Need pandas-specific operations not in Spark (e.g., `iterrows`) | Convert to pandas with `.to_pandas()` on a small subset |
| Need maximum performance and fine-grained control | **Use native PySpark** — avoid pyspark.pandas overhead |

### Operations that are expensive in pyspark.pandas

```python
# Expensive — triggers full data collection or global sort:
psdf.sort_values('col')       # global sort = full shuffle
psdf.to_pandas()              # full collect to driver
len(psdf)                     # count action = full scan
psdf.iterrows()               # row-by-row iteration = catastrophically slow
psdf.index                    # may trigger full scan depending on index type

# Prefer:
psdf.sort_values('col').head(10)  # still sorts but bounded output
psdf.count()                      # same cost as len() but more explicit
# Never use iterrows() — use vectorised operations instead
```

### Index types and performance

```python
# Configure via spark option:
ps.set_option('compute.default_index_type', 'distributed')     # fastest, not globally ordered
ps.set_option('compute.default_index_type', 'sequence')        # sequential but slow (requires full scan)
ps.set_option('compute.default_index_type', 'distributed-sequence')  # globally sequential, extra shuffle

# For most workloads: 'distributed' is the best choice
# Only use 'sequence' when exact sequential index values are required
```

---

## Real-World Scenarios

<details>
<summary>Scenario 1: Migrating a pandas data pipeline to run on 100 GB instead of 1 GB</summary>

**Situation:**
A data analyst has a working pandas pipeline that processes 1 GB CSV files. The data has grown to 100 GB. The company has a Spark cluster. The analyst wants to scale with minimal code changes.

**Original pandas code:**
```python
import pandas as pd

df = pd.read_csv('/data/sales.csv')
result = df.groupby('region').agg({'revenue': 'sum', 'orders': 'count'})
result = result.sort_values('revenue', ascending=False)
result.to_csv('/output/region_summary.csv', index=False)
```

**Migrated to pyspark.pandas — minimal change:**
```python
import pyspark.pandas as ps  # ← only change this import

df = ps.read_csv('/data/sales.csv')  # now reads 100 GB distributed
result = df.groupby('region').agg({'revenue': 'sum', 'orders': 'count'})
result = result.sort_values('revenue', ascending=False)
result.to_csv('/output/region_summary.csv', index=False)  # writes distributed
# Code is nearly identical — just the import changed!
```

**Key insight:** For many standard pandas operations, `import pyspark.pandas as ps` is the only code change needed. The API is intentionally compatible.

**Exam Sub-topic:** Migration from pandas to pyspark.pandas; `import pyspark.pandas as ps` pattern; read/write operations
</details>

<details>
<summary>Scenario 2: Converting between APIs — when to use to_pandas() vs to_spark()</summary>

**Situation:**
A pipeline reads large data via Spark, uses pyspark.pandas for processing, then needs to: (a) save results as Parquet and (b) create a small summary report using a pandas-only library (matplotlib).

**Solution:**
```python
import pyspark.pandas as ps
from pyspark.sql import SparkSession
import matplotlib.pyplot as plt

spark = SparkSession.builder.getOrCreate()

# Step 1: Read large data via Spark DataFrame
sdf_raw = spark.read.parquet('/data/transactions/')  # 10 GB of data

# Step 2: Convert to pyspark.pandas for familiar API
psdf = sdf_raw.pandas_api()  # no data movement — just API change

# Step 3: Process using pandas-style API
psdf_summary = psdf.groupby('product_category').agg({'revenue': 'sum'}).reset_index()

# Step 4a: Save full results as Parquet — use to_spark() for efficiency
psdf_summary.to_spark().write.mode('overwrite').parquet('/output/category_revenue/')
# No collect — stays distributed

# Step 4b: Create a bar chart — use to_pandas() on the SMALL summary (6 rows)
pdf_chart = psdf_summary.to_pandas()  # only 6 rows — safe to collect
pdf_chart.plot.bar(x='product_category', y='revenue')
plt.savefig('revenue_chart.png')
```

**Rule:** Use `.to_spark()` to pass data to distributed systems. Use `.to_pandas()` ONLY when the result is already small.

**Exam Sub-topic:** Conversion methods; `.to_pandas()` collect warning; `.to_spark()` no data movement; `.pandas_api()` no data movement
</details>

<details>
<summary>Scenario 3: Understanding why iterrows() is dangerous in pyspark.pandas</summary>

**Situation:**
A developer copies a pattern from native pandas — using `iterrows()` to process each row — into a pyspark.pandas script on 50 million rows.

**Dangerous pattern:**
```python
# Native pandas — tolerable on small data:
for idx, row in pdf.iterrows():
    print(row['name'], row['salary'] * 1.1)  # slow but works on small data

# pyspark.pandas — CATASTROPHICALLY slow on large data:
for idx, row in psdf.iterrows():  # ← DO NOT USE
    print(row['name'], row['salary'] * 1.1)
    # Each iteration collects one row from the Spark cluster to the driver
    # For 50M rows: 50 million round trips to the cluster
    # Will take hours or days
```

**Correct pattern — vectorised operations:**
```python
# Use apply() or column operations instead:
psdf['adjusted_salary'] = psdf['salary'] * 1.1  # vectorised — single Spark job

# Or use apply() for custom logic:
psdf['adjusted_salary'] = psdf['salary'].apply(lambda s: s * 1.1)

# Show results:
psdf[['name', 'salary', 'adjusted_salary']].show()
```

**Exam Sub-topic:** Behavioral differences from native pandas; `iterrows()` is expensive; always use vectorised operations; pandas-to-Spark migration pitfalls
</details>

<details>
<summary>Scenario 4: Choosing between pyspark.pandas and native PySpark for a new ETL pipeline</summary>

**Situation:**
A data engineer is building a new ETL pipeline from scratch to process 500 GB of daily transaction data. They need to decide: use `pyspark.pandas` or native PySpark DataFrame API?

**Analysis:**

| Factor | pyspark.pandas | Native PySpark |
|--------|---------------|----------------|
| Code familiarity | pandas-like — easy for pandas users | Spark-specific API |
| Performance overhead | Some overhead from API translation layer | Minimal — direct Catalyst optimisation |
| Partition control | Limited — mostly hidden | Full control (`repartition`, `coalesce`) |
| Complex transformations | Some operations unsupported or slow | All operations supported |
| Code verbosity | Less verbose for common operations | More verbose but explicit |
| Production recommendation | For migration/prototyping | **For new production pipelines** |

**Recommendation for new code:** Use native PySpark DataFrame API for production ETL pipelines. Use `pyspark.pandas` when migrating existing pandas code.

```python
# Native PySpark — recommended for new production pipeline:
from pyspark.sql.functions import col, sum as _sum, avg, date_trunc

df = spark.read.parquet('/data/transactions/')
daily_summary = df \
    .withColumn('day', date_trunc('day', col('txn_ts'))) \
    .groupBy('day', 'merchant_category') \
    .agg(
        _sum('amount').alias('total_amount'),
        avg('amount').alias('avg_amount'),
    ) \
    .repartition(10, col('merchant_category'))  # explicit partition control

daily_summary.write.partitionBy('day').parquet('/output/daily_summary/')
```

**Exam Sub-topic:** When to use pyspark.pandas vs native PySpark; performance trade-offs; migration vs new development
</details>

<details>
<summary>Scenario 5: Three-way comparison — the same join written in all three APIs</summary>

**Situation:**
A developer needs to join a 10 GB transactions table with a small 500-row lookup table. Here is the same join written three ways.

**API 1 — Native pandas (single machine only):**
```python
import pandas as pd

txns = pd.read_csv('/data/transactions.csv')    # must fit in RAM
lookup = pd.read_csv('/data/categories.csv')

result = txns.merge(lookup, on='category_id', how='left')
result.to_csv('/output/enriched.csv', index=False)
```

**API 2 — pyspark.pandas (distributed, pandas-style):**
```python
import pyspark.pandas as ps

txns = ps.read_csv('/data/transactions.csv')    # distributed — can be 10 GB
lookup = ps.read_csv('/data/categories.csv')

result = txns.merge(lookup, on='category_id', how='left')
result.to_csv('/output/enriched.csv', index=False)
```

**API 3 — Native PySpark (distributed, Spark-style):**
```python
from pyspark.sql.functions import broadcast

txns = spark.read.csv('/data/transactions.csv', header=True, inferSchema=True)
lookup = spark.read.csv('/data/categories.csv', header=True, inferSchema=True)

# Use broadcast join since lookup is small — pyspark.pandas does this automatically
result = txns.join(broadcast(lookup), on='category_id', how='left')
result.write.mode('overwrite').csv('/output/enriched.csv', header=True)
```

**Key insight:** pyspark.pandas `merge()` maps to a Spark join. For the small lookup table, pyspark.pandas may automatically apply a broadcast join optimisation. With native PySpark you can explicitly force it with `broadcast()`.

**Exam Sub-topic:** `merge()` in pyspark.pandas vs `join()` in PySpark; three-way API comparison; `broadcast()` for small table optimisation
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| Import statement | `import pyspark.pandas as ps` |
| Previously called | **Koalas** (merged into PySpark in Spark 3.2) |
| Conventional alias | `ps` |
| Create from dict | `ps.DataFrame({'col': [values]})` |
| Create from CSV | `ps.read_csv('/path/to/file.csv')` |
| Create from native pandas | `ps.from_pandas(pdf)` |
| Spark DataFrame → Spark pandas | `sdf.pandas_api()` or `ps.from_spark(sdf)` |
| Spark pandas → Native pandas | `psdf.to_pandas()` — **CAUTION: full collect** |
| Spark pandas → Spark DataFrame | `psdf.to_spark()` |
| Does `.to_pandas()` trigger a collect? | **Yes** — collects ALL data to driver |
| Does `.to_spark()` trigger a collect? | **No** — no data movement |
| Does `.pandas_api()` trigger a collect? | **No** — no data movement |
| Row ordering guaranteed? | **No** — no ordering without explicit `sort_values()` |
| In-place operations supported? | **No** — must reassign: `psdf = psdf.drop(...)` |
| Index type default | May be distributed (unstable) — configure via `ps.set_option` |
| `iterrows()` in pyspark.pandas | **Very expensive** — avoid; use vectorised ops |
| Pandas `merge()` → Spark equivalent | `join()` |
| When to use pyspark.pandas | Migrating existing pandas code; data scientists |
| When to use native PySpark | New production pipelines; need full control |
| SparkSession needed? | **Yes** — pyspark.pandas uses the active SparkSession |

---

## Practice Q&A

<details>
<summary>Q1: What is the correct import for the Pandas API on Apache Spark?</summary>

**A:**
```python
import pyspark.pandas as ps
```

**Not:**
```python
import pandas as pd         # ← native pandas (single machine)
from koalas import pandas   # ← old Koalas import (deprecated)
import pyspark.pandas as pd # ← wrong alias — should be ps
```

The conventional alias is `ps`. The module lives at `pyspark.pandas` (not `koalas` — Koalas was merged into PySpark as of Spark 3.2).
</details>

<details>
<summary>Q2: What are the four ways to create a Spark pandas DataFrame, and what is the warning for to_pandas()?</summary>

**A:**

```python
import pyspark.pandas as ps

# 1. From Python dict
psdf = ps.DataFrame({'a': [1, 2, 3], 'b': ['x', 'y', 'z']})

# 2. From CSV
psdf = ps.read_csv('/path/to/data.csv')

# 3. From native pandas
psdf = ps.from_pandas(pdf)

# 4. From Spark DataFrame
psdf = spark_df.pandas_api()
# OR:
psdf = ps.from_spark(spark_df)
```

**Warning for `.to_pandas()`:** This method **collects ALL data** from the distributed Spark cluster to the driver machine's memory. It should only be used when the resulting DataFrame is small. Calling `.to_pandas()` on a DataFrame with millions of rows will trigger an OOM error on the driver.
</details>

<details>
<summary>Q3: What are three key behavioral differences between native pandas and pyspark.pandas?</summary>

**A:**

1. **Index behavior** — Native pandas has a stable sequential RangeIndex. Spark pandas has a distributed index that may not be globally sequential. Configure with `ps.set_option('compute.default_index_type', ...)`. Default may be unstable.

2. **No guaranteed row ordering** — Native pandas preserves insertion order. Spark pandas has no ordering guarantee because data is distributed. Always use `sort_values()` when order matters.

3. **No in-place operations** — Native pandas supports `df.drop(..., inplace=True)`. Spark pandas DataFrames are immutable — operations return a new DataFrame: `psdf = psdf.drop('col', axis=1)` (must reassign).

**Bonus:** `iterrows()` is extremely expensive in pyspark.pandas — triggers row-by-row data collection from the cluster. Never use it; prefer vectorised operations.
</details>

<details>
<summary>Q4: When should you use pyspark.pandas vs the native PySpark DataFrame API?</summary>

**A:**

**Use pyspark.pandas when:**
- **Migrating existing pandas code** to run on larger data — minimal code changes required
- The team is more familiar with pandas syntax than Spark
- Rapid prototyping where exact performance is not critical
- Data scientists running ad-hoc analyses on large datasets

**Use native PySpark DataFrame API when:**
- **Building new production pipelines** from scratch — no migration overhead
- Need explicit control over partitioning (`repartition`, `coalesce`)
- Maximum performance is required — avoid the pyspark.pandas translation layer
- Using operations not well-supported by pyspark.pandas
- Explicit broadcast joins, custom aggregations, or streaming

**General rule:** If you already have pandas code, use pyspark.pandas to scale it. If starting fresh, use the native DataFrame API.
</details>

<details>
<summary>Q5: Write the same groupby aggregation using all three APIs — native pandas, pyspark.pandas, and native PySpark.</summary>

**A:**

**Objective:** Sum of `revenue` and count of `orders`, grouped by `region`, sorted descending.

**Native pandas:**
```python
import pandas as pd

result = df.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    order_count=('orders', 'count')
).reset_index().sort_values('total_revenue', ascending=False)
```

**pyspark.pandas:**
```python
import pyspark.pandas as ps

result = psdf.groupby('region').agg(
    total_revenue=('revenue', 'sum'),
    order_count=('orders', 'count')
).reset_index().sort_values('total_revenue', ascending=False)
# Syntax nearly identical to native pandas
```

**Native PySpark:**
```python
from pyspark.sql.functions import sum as _sum, count, desc

result = sdf.groupBy('region') \
    .agg(
        _sum('revenue').alias('total_revenue'),
        count('orders').alias('order_count')
    ) \
    .orderBy(desc('total_revenue'))
# Different syntax: groupBy() vs groupby(), orderBy() vs sort_values()
```
</details>